# Assay Evidence Studio · Heavyweight AI / Deep Learning / RL Reference Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tun539/assay-evidence-lab/blob/main/ml/BioAI_PyTorch_ML_RL_Suite.ipynb)

This companion engineering notebook connects **Assay Evidence Studio**'s browser workbenches (`Molecular Learning Lab`, `Gene Expression Atlas`, `Kinetics & OOD`, `Simulation & BioEmu`, and `Cellpose & AlphaFold Audit`) directly to reproducible **PyTorch Deep Learning**, **Classical Machine Learning**, and **Reinforcement Learning (Tabular Q, Double DQN, and Continuous-Dose PPO-Clip)** pipelines.

### Architectural Pillars Covered
1. **Multi-Omics & Studio Bundle Ingestion**: Direct loaders parsing primary production assets (`studio/expression-data.js` 7,129 Golub probes × 34 test specimens, `studio/research-data.js` 569 nested CV cytology specimens, `studio/folds/P04637.js` TP53 AlphaFold Cα/pLDDT/PAE, and simulated Michaelis–Menten enzyme kinetics).
2. **Classical ML & Anti-Leakage Auditing**: Strict `sklearn.pipeline.Pipeline` cross-validation (Gaussian NB, Shrinkage LDA, Random Forest, HistGradientBoosting) + quantitative proof of **in-sample selection leakage inflation**.
3. **Heavyweight PyTorch Deep Learning**:
   - `KinResNet`: Heteroscedastic Residual MLP (`LayerNorm` + `SiLU` + `GaussianNLLLoss` + **MC Dropout** epistemic vs aleatoric variance) tested on In-Domain vs Out-of-Distribution (`[S]`) extrapolation.
   - `GolubAttentionNet`: Self-Attention Gene Signature Encoder with `BCEWithLogitsLoss` + Input×Gradient biomarker saliency attribution (`CST3`, `CD33`, `HOXA9`, `TP53`).
   - `PredictedLDDTCalibrator` & **Conservative Neural Potential (`SmoothNet`, $F_\theta = -\nabla_x U_\theta$)** audited for Expected Calibration Error (`ECE`) via L-BFGS and symplectic Velocity Verlet energy drift.
4. **Heavyweight Reinforcement Learning for Sequential Assay Discovery**:
   - Compares **Tabular Q-Learning / UCB1**, **PyTorch Double DQN** (Replay Buffer + Polyak target updates + Huber loss), and **Continuous-Dose PPO-Clip (`Beta` Policy + GAE-λ + Fisher Information D-optimality)** on a multi-step active experimentation MDP.
5. **Product Bridge (`assay-evidence/1`)**: Exports benchmark records directly as a schema-valid JSON packet ready to import into `studio/workspace.html`.

In [ ]:
# Cell 1 · Environment & Repository Ingestion (Local or Google Colab)
from __future__ import annotations

import json
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import Ridge
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Beta

def locate_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent]:
        if (cand / "studio" / "expression-data.js").is_file():
            return cand
    colab_clone = Path("/content/assay-evidence-lab")
    if not colab_clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tun539/assay-evidence-lab.git", str(colab_clone)], check=True)
    return colab_clone

ROOT = locate_repo_root()
sys.path.insert(0, str(ROOT / "ml"))
import molecular as lab  # Bundled Michaelis-Menten & tabular Q reference module

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

def load_js_json(rel_path: str, prefix: str) -> dict:
    raw = (ROOT / rel_path).read_text(encoding="utf-8")
    body = raw.split("=", 1)[1].strip()
    if body.endswith(";"):
        body = body[:-1]
    return json.loads(body)

expr_bundle = load_js_json("studio/expression-data.js", "expression")
research_bundle = load_js_json("studio/research-data.js", "benchmark")
tp53_fold = load_js_json("studio/folds/P04637.js", "bundledFolds")

print(f"Repository root: {ROOT} | PyTorch {torch.__version__} on {DEVICE}")
print(f"Golub bundle: {len(expr_bundle['probes'])} probes, {len(expr_bundle['selected'])} selected, {len(expr_bundle['ids'])} test patients")
print(f"WDBC bundle: {len(research_bundle['rows'])} specimens across 5 stratified folds")
print(f"AlphaFold TP53 P04637: {len(tp53_fold['rows'])} residues, mean pLDDT {tp53_fold['plddtMean']:.2f}, PAE {tp53_fold['paeSize']}x{tp53_fold['paeSize']}")

## Chapter 1 · Classical Machine Learning & Data-Leakage Audit

We establish rigorous non-deep baselines on two core product datasets:
1. **Enzyme Kinetics Parameter Recovery (2,400 train / 800 test curves, plus OOD stress test)**: Ridge (`α=100`), Random Forest, HistGradientBoostingRegressor, and exact SciPy `least_squares`.
2. **Golub Leukemia Microarray (`7,129` probes) & WDBC (`569` specimens)**: Verifies zero-leakage feature selection vs simulated leaky peeking.

In [ ]:
# Cell 2 · Classical ML Kinetics Benchmark & In-Domain vs OOD Audit
params_tr, rates_tr, fam_tr = lab.simulate(2400, 17)
params_te, rates_te, fam_te = lab.simulate(800, 91)

y_tr = np.column_stack([params_tr[:, 0] / 180.0, np.log(params_tr[:, 1] / 0.1)])
scaler_kin = StandardScaler().fit(rates_tr)
x_tr_s = scaler_kin.transform(rates_tr)
x_te_s = scaler_kin.transform(rates_te)

ridge_m = Ridge(alpha=100.0).fit(x_tr_s, y_tr)
rf_m = RandomForestRegressor(n_estimators=160, min_samples_leaf=3, random_state=SEED, n_jobs=-1).fit(rates_tr, y_tr)
hgb_v = HistGradientBoostingRegressor(max_iter=200, max_depth=6, random_state=SEED).fit(rates_tr, y_tr[:, 0])
hgb_k = HistGradientBoostingRegressor(max_iter=200, max_depth=6, random_state=SEED).fit(rates_tr, y_tr[:, 1])

def unscale_targets(pred_2d: np.ndarray) -> np.ndarray:
    return np.column_stack([pred_2d[:, 0] * 180.0, 0.1 * np.exp(np.clip(pred_2d[:, 1], -5.0, 5.0))])

def eval_kinetics_model(name: str, pred_2d: np.ndarray, truth: np.ndarray) -> dict:
    est = unscale_targets(pred_2d)
    mae = np.abs(est - truth).mean(axis=0)
    # OOD rate reconstruction error at held-out high endpoint 2.5 ppm
    extrap_s = 2.5
    true_extrap = truth[:, 0] * extrap_s / (truth[:, 1] + extrap_s)
    pred_extrap = est[:, 0] * extrap_s / (est[:, 1] + extrap_s)
    ood_rmse = float(np.sqrt(np.mean((pred_extrap - true_extrap) ** 2)))
    return {"Model": name, "Vmax MAE": mae[0], "Km MAE": mae[1], "OOD [2.5 ppm] RMSE": ood_rmse}

kin_results = [
    eval_kinetics_model("Ridge (alpha=100)", ridge_m.predict(x_te_s), params_te),
    eval_kinetics_model("Random Forest (160 trees)", rf_m.predict(rates_te), params_te),
    eval_kinetics_model("HistGradientBoosting", np.column_stack([hgb_v.predict(rates_te), hgb_k.predict(rates_te)]), params_te),
]
print(pd.DataFrame(kin_results).set_index("Model").round(4))

# Golub Cross-Cohort & WDBC Leakage Audit
probes = expr_bundle["probes"]
tr_diff = np.array([p[3] - p[2] for p in probes], dtype=float)
te_diff = np.array([p[5] - p[4] for p in probes], dtype=float)
sel_idx = np.array(expr_bundle["selected"], dtype=int)
r_genome = float(np.corrcoef(tr_diff, te_diff)[0, 1])
r_panel = float(np.corrcoef(tr_diff[sel_idx], te_diff[sel_idx])[0, 1])

top_leaky = np.argsort(np.abs(tr_diff + te_diff))[::-1][:25]
clean_abs = float(np.abs(te_diff[sel_idx]).mean())
leaky_abs = float(np.abs(te_diff[top_leaky]).mean())
print(f"\nGolub cross-cohort replication: genome-wide r={r_genome:.3f} | 25-probe ANOVA panel r={r_panel:.3f}")
print(f"Leakage test-contrast inflation factor: {leaky_abs / clean_abs:.2f}x (Clean={clean_abs:.1f} vs Leaky={leaky_abs:.1f})")
# Classical ML Classifier Benchmark on Golub Cohort Profiles using Strict Pipeline
clf_bench = []
for cname, est in [
    ("Gaussian Naive Bayes", GaussianNB()),
    ("Shrinkage LDA", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ("Random Forest (200 trees)", RandomForestClassifier(n_estimators=200, random_state=SEED)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=120, max_depth=4, random_state=SEED)),
]:
    pipe = make_pipeline(SelectKBest(f_classif, k=min(25, len(sel_idx))), StandardScaler(), est)
    # Build cohort-matched training/test evaluation over all 25 selected Golub probes
    tr_mat_0 = np.array([p[2] for p in probes])[None, :] + np.random.default_rng(11).normal(0, 120, (40, len(probes)))
    tr_mat_1 = np.array([p[3] for p in probes])[None, :] + np.random.default_rng(12).normal(0, 120, (40, len(probes)))
    te_mat_0 = np.array([p[4] for p in probes])[None, :] + np.random.default_rng(13).normal(0, 140, (30, len(probes)))
    te_mat_1 = np.array([p[5] for p in probes])[None, :] + np.random.default_rng(14).normal(0, 140, (30, len(probes)))
    pipe.fit(np.vstack([tr_mat_0, tr_mat_1]), np.concatenate([np.zeros(40), np.ones(40)]))
    x_ev = np.vstack([te_mat_0, te_mat_1]); y_ev = np.concatenate([np.zeros(30), np.ones(30)])
    pr = pipe.predict_proba(x_ev)[:, 1]
    clf_bench.append({"Classifier": cname, "ROC-AUC": roc_auc_score(y_ev, pr), "PR-AP": average_precision_score(y_ev, pr), "BalAcc": balanced_accuracy_score(y_ev, pipe.predict(x_ev))})
print("\n7,129-Probe Strict Pipeline Comparison:")
print(pd.DataFrame(clf_bench).set_index("Classifier").round(4))
print("\nOfficial Precomputed Golub 34-Specimen Test Cohort:")
print(pd.DataFrame([{"Model": m["name"], "AUROC": m["auc"], "BalancedAcc": m["ba"]} for m in expr_bundle["models"]]).set_index("Model").round(4))

## Chapter 2 · Heavyweight PyTorch Deep Learning Systems

### 2A · `KinResNet`: Heteroscedastic Probabilistic Residual MLP + MC Dropout Uncertainty
Naive MLPs emit point estimates without knowing *when* a prediction sits outside training support. `KinResNet` predicts **both** parameter location $\mu = (V_{\max}/180, \log(K_m/0.1))$ and aleatoric log-variance $\log \sigma^2$ trained via `nn.GaussianNLLLoss`, paired with **MC Dropout (`T=30` stochastic passes)** to separate irreducible noise from structural uncertainty on extreme enzyme families.

In [ ]:
# Cell 3 · PyTorch KinResNet (Residual Blocks + LayerNorm + SiLU + Heteroscedastic NLL + MC Dropout)
class ResBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.12):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.net(x)


class KinResNet(nn.Module):
    def __init__(self, in_dim: int = 6, hidden: int = 128, n_blocks: int = 3):
        super().__init__()
        self.stem = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([ResBlock(hidden, dropout=0.10) for _ in range(n_blocks)])
        self.head_mu = nn.Sequential(nn.LayerNorm(hidden), nn.SiLU(), nn.Linear(hidden, 2))
        self.head_logvar = nn.Sequential(nn.LayerNorm(hidden), nn.SiLU(), nn.Linear(hidden, 2))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.stem(x)
        for blk in self.blocks:
            h = blk(h)
        mu = self.head_mu(h)
        logvar = self.head_logvar(h).clamp(-6.0, 2.5)
        return mu, logvar


def train_kinresnet(x_np: np.ndarray, y_np: np.ndarray, epochs: int = 180) -> KinResNet:
    model = KinResNet(in_dim=x_np.shape[1], hidden=128, n_blocks=3).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2.5e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.GaussianNLLLoss()

    x_t = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y_np, dtype=torch.float32, device=DEVICE)
    ds = torch.utils.data.TensorDataset(x_t, y_t)
    loader = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            mu, logvar = model(xb)
            loss = loss_fn(mu, yb, torch.exp(logvar))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        sched.step()
    return model


@torch.no_grad()
def mc_dropout_predict(model: KinResNet, x_np: np.ndarray, n_passes: int = 30):
    x_t = torch.tensor(x_np, dtype=torch.float32, device=DEVICE)
    model.train()  # Keep Dropout active for epistemic uncertainty sampling
    mus, vars_ = [], []
    for _ in range(n_passes):
        mu, logvar = model(x_t)
        mus.append(mu.cpu().numpy())
        vars_.append(np.exp(logvar.cpu().numpy()))
    mus_arr = np.stack(mus, axis=0)
    mu_mean = mus_arr.mean(axis=0)
    epistemic_std = mus_arr.std(axis=0)
    aleatoric_std = np.sqrt(np.stack(vars_, axis=0).mean(axis=0))
    model.eval()
    return mu_mean, epistemic_std, aleatoric_std


kin_net = train_kinresnet(x_tr_s, y_tr, epochs=180)
mu_te, epi_te, ale_te = mc_dropout_predict(kin_net, x_te_s, n_passes=30)
kinresnet_row = eval_kinetics_model("PyTorch KinResNet (Heteroscedastic + MC)", mu_te, params_te)
print("KinResNet Benchmark:", pd.Series(kinresnet_row).to_dict())
print(f"Mean Epistemic std: {epi_te.mean(axis=0).round(4)} | Mean Aleatoric std: {ale_te.mean(axis=0).round(4)}")

### 2B · PyTorch `GolubAttentionNet` (Self-Attention Biomarker Classifier & Gradient Attribution)
Rather than treating probes independently like Naive Bayes, we embed each selected transcript intensity alongside learnable pathway/probe embeddings, apply multi-head Self-Attention across all `K=25` panel biomarkers, and extract **Input × Gradient saliency** to quantify each gene's causal influence (`CST3`, `CD33`, `HOXA9`, `AZU1`, `LYZ`). Since raw Affymetrix per-patient CSVs live upstream, we generate cohort-calibrated synthetic patients matching the exact Golub cohort means/dispersion for training and audit on the 34 official Golub decision scores.

In [ ]:
# Cell 4 · PyTorch Gated-Attention Classifier & Biomarker Saliency over 25-Gene Signatures
class GolubAttentionNet(nn.Module):
    def __init__(self, n_genes: int = 25, d_model: int = 32, n_heads: int = 4):
        super().__init__()
        self.gene_emb = nn.Parameter(torch.randn(1, n_genes, d_model) * 0.08)
        self.val_proj = nn.Linear(1, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=0.15, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x: (B, n_genes) standardized gene profile
        tok = self.val_proj(x.unsqueeze(-1)) + self.gene_emb
        att_out, _ = self.attn(tok, tok, tok)
        tok = self.norm1(tok + att_out)
        tok = self.norm2(tok + self.ffn(tok))
        w = torch.softmax(self.gate(tok).squeeze(-1), dim=1)  # (B, n_genes) attention weights
        pooled = (tok * w.unsqueeze(-1)).sum(dim=1)
        logit = self.head(pooled).squeeze(-1)
        return logit, w


panel_rows = [probes[i] for i in sel_idx]
rng = np.random.default_rng(SEED)
tr_means_all = np.array([r[2] for r in panel_rows])
tr_means_aml = np.array([r[3] for r in panel_rows])
te_means_all = np.array([r[4] for r in panel_rows])
te_means_aml = np.array([r[5] for r in panel_rows])
feat_scale = np.maximum(250.0, 0.35 * (np.abs(tr_means_all) + np.abs(tr_means_aml)))

def sample_cohort_patients(n_all: int, n_aml: int, m0: np.ndarray, m1: np.ndarray, sd: np.ndarray):
    x0 = m0[None, :] + rng.normal(0.0, 1.0, size=(n_all, len(m0))) * sd[None, :]
    x1 = m1[None, :] + rng.normal(0.0, 1.0, size=(n_aml, len(m1))) * sd[None, :]
    x = np.vstack([x0, x1])
    y = np.concatenate([np.zeros(n_all), np.ones(n_aml)])
    return x, y

x_g_tr, y_g_tr = sample_cohort_patients(160, 110, tr_means_all, tr_means_aml, feat_scale)
x_g_te, y_g_te = sample_cohort_patients(80, 56, te_means_all, te_means_aml, feat_scale)
g_scaler = StandardScaler().fit(x_g_tr)
xg_tr_t = torch.tensor(g_scaler.transform(x_g_tr), dtype=torch.float32, device=DEVICE)
yg_tr_t = torch.tensor(y_g_tr, dtype=torch.float32, device=DEVICE)
xg_te_t = torch.tensor(g_scaler.transform(x_g_te), dtype=torch.float32, device=DEVICE)

attn_net = GolubAttentionNet(n_genes=len(sel_idx), d_model=32, n_heads=4).to(DEVICE)
opt_attn = torch.optim.AdamW(attn_net.parameters(), lr=3e-3, weight_decay=5e-4)
for _ in range(110):
    attn_net.train()
    opt_attn.zero_grad()
    logits, _ = attn_net(xg_tr_t)
    loss = F.binary_cross_entropy_with_logits(logits, yg_tr_t)
    loss.backward()
    opt_attn.step()

attn_net.eval()
xg_te_req = xg_te_t.clone().detach().requires_grad_(True)
logits_te, attn_w = attn_net(xg_te_req)
loss_attr = logits_te.sum()
loss_attr.backward()
saliency = (xg_te_req.grad * xg_te_req).abs().mean(dim=0).detach().cpu().numpy()
probs_attn = torch.sigmoid(logits_te).detach().cpu().numpy()

top_saliency = np.argsort(saliency)[::-1][:8]
print(f"GolubAttentionNet Eval ROC-AUC: {roc_auc_score(y_g_te, probs_attn):.4f} | Brier: {brier_score_loss(y_g_te, probs_attn):.4f}")
print("Top Biomarker Attributions:", [(panel_rows[k][0], round(float(saliency[k]), 4)) for k in top_saliency])

### 2C · AlphaFold `PredictedLDDTHead` Temperature Calibration ($T^*$) & Conservative Neural Potential ($F = -\nabla_x U_\theta$)
Two critical safety audits from `audit.html` and `simulation.html`:
1. **Temperature Calibration (`ECE`)** via PyTorch **L-BFGS** on AlphaFold 50-bin confidence logits around real `TP53 (P04637)` per-residue pLDDT values.
2. **$C^\infty$ Conservative Neural Potential (`SmoothNet`)**: Verifies why differentiating a scalar energy $U_\theta(x)$ with `torch.autograd.grad` conserves Hamiltonian energy during Velocity Verlet MD whereas unconstrained force regression accumulates exponential energy drift.

In [ ]:
# Cell 5 · AlphaFold pLDDT Calibration (L-BFGS) & Conservative Neural Potential Verlet Conservation
# 1) Construct synthetic 50-bin logits matching TP53 P04637 empirical pLDDT distribution (miscalibrated T_raw=0.55)
plddt_vals = np.array([r[5] for r in tp53_fold["rows"]], dtype=np.float32)
bin_centers = np.linspace(1.0, 99.0, 50, dtype=np.float32)
dist_matrix = -((plddt_vals[:, None] - bin_centers[None, :]) ** 2) / (2 * 6.5**2)
raw_logits = torch.tensor(dist_matrix * 1.8, dtype=torch.float32, device=DEVICE)
is_confident = torch.tensor((plddt_vals >= 70.0).astype(np.float32), device=DEVICE)

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for b in range(n_bins):
        mask = (probs >= edges[b]) & (probs <= edges[b + 1] if b == n_bins - 1 else probs < edges[b + 1])
        if mask.any():
            ece += (mask.mean()) * abs(labels[mask].mean() - probs[mask].mean())
    return float(ece)

with torch.no_grad():
    p_uncal = torch.softmax(raw_logits, dim=-1)[:, bin_centers >= 70.0].sum(dim=-1).cpu().numpy()
ece_before = compute_ece(p_uncal, is_confident.cpu().numpy())

log_temp = nn.Parameter(torch.zeros(1, device=DEVICE))
lbfgs = torch.optim.LBFGS([log_temp], lr=0.25, max_iter=40)
def closure():
    lbfgs.zero_grad()
    T = torch.exp(log_temp)
    probs = torch.softmax(raw_logits / T, dim=-1)[:, bin_centers >= 70.0].sum(dim=-1).clamp(1e-6, 1 - 1e-6)
    loss = F.binary_cross_entropy(probs, is_confident)
    loss.backward()
    return loss
lbfgs.step(closure)
T_opt = float(torch.exp(log_temp).item())
with torch.no_grad():
    p_cal = torch.softmax(raw_logits / T_opt, dim=-1)[:, bin_centers >= 70.0].sum(dim=-1).cpu().numpy()
ece_after = compute_ece(p_cal, is_confident.cpu().numpy())
print(f"AlphaFold TP53 L-BFGS Calibration: Optimal T* = {T_opt:.3f} | ECE before={ece_before:.4f} -> after={ece_after:.4f}")

# 2) Conservative Neural Potential: F_theta(x) = -dU_theta/dx vs Reference Morse + Quartic Potential
class SmoothPotential(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 32), nn.Softplus(), nn.Linear(32, 32), nn.Softplus(), nn.Linear(32, 1))

    def energy_and_force(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x_req = x.clone().detach().requires_grad_(True)
        u = self.net(x_req).sum(-1)
        du_dx = torch.autograd.grad(u.sum(), x_req, create_graph=True)[0]
        return u, -du_dx

# Reference anharmonic double-well / quartic surface U(x) = 0.25 x^4 - 0.8 x^2 + 0.1 x
xs_train = torch.linspace(-2.2, 2.2, 400, device=DEVICE).unsqueeze(-1)
f_true = -(xs_train**3 - 1.6 * xs_train + 0.1)
pot_net = SmoothPotential().to(DEVICE)
opt_pot = torch.optim.AdamW(pot_net.parameters(), lr=8e-3)
for _ in range(320):
    opt_pot.zero_grad()
    _, f_pred = pot_net.energy_and_force(xs_train)
    loss_f = F.mse_loss(f_pred, f_true)
    loss_f.backward()
    opt_pot.step()

# Integrate 250 Velocity Verlet steps with trained Conservative Neural Potential
dt = 0.025
pos, vel = -1.45, 0.35
energies = []
for _ in range(250):
    u0, f0 = pot_net.energy_and_force(torch.tensor([[pos]], device=DEVICE))
    f0_val = float(f0.item())
    energies.append(float(u0.item()) + 0.5 * vel**2)
    pos = pos + vel * dt + 0.5 * f0_val * dt**2
    _, f1 = pot_net.energy_and_force(torch.tensor([[pos]], device=DEVICE))
    vel = vel + 0.5 * (f0_val + float(f1.item())) * dt

e_drift = np.max(np.abs(np.array(energies) - energies[0]))
print(f"Conservative Neural Force Field (250 Verlet steps, dt={dt}): max Hamiltonian |Delta H| = {e_drift:.5f}")

## Chapter 3 · Heavyweight Reinforcement Learning for Active Biochemical Experimentation

Why stop at passive curve fitting? In real Bio-AI discovery platforms, wet-lab assays cost money and time per measurement. Here we formulate sequential substrate experimentation as a formal MDP (`ActiveAssayEnv`) and pit **three RL algorithms** head-to-head:
1. **Tabular Q-Learning & UCB1** (exact reproduction of `molecular.py`'s `17×5` policy heatmap).
2. **PyTorch Double Deep Q-Network (`DoubleDQN`)** operating over continuous state embeddings `(anchor_rate, last_action, last_rate, step_frac, normalized_posterior)` with Experience Replay (`20,000`), Polyak Target Updates ($\tau=0.01$), and Huber Loss across `T=2` sequential follow-up doses.
3. **Continuous-Dose `PPOActorCritic` (`Beta` Policy + `GAE-λ` + Fisher D-Optimality Shaping)** choosing continuous substrate concentrations $a_t \in [0.02, 1.20]\text{ ppm}$ trained with `L_CLIP` surrogate updates.

In [ ]:
# Cell 6 · Tabular Q-Learning Parity + Sequential Multi-Step Active Assay Environment
_, ep_rates, ep_fam = lab.simulate(4800, 42)
_, test_rates, test_truth = lab.simulate(1200, 91)

# Baseline 1-step Tabular Q (exact Studio Molecular Learning Lab policy)
tab_rewards = np.column_stack([lab.decoder(rates_tr, fam_tr, ep_rates, a) == ep_fam for a in lab.ACTIONS])
q_tab = lab.train_policy(np.digitize(ep_rates[:, lab.ANCHOR], lab.BINS), tab_rewards)
test_scores = np.column_stack([lab.decoder(rates_tr, fam_tr, test_rates, a) == test_truth for a in lab.ACTIONS])
tab_acc = float(test_scores[np.arange(len(test_rates)), q_tab[np.digitize(test_rates[:, lab.ANCHOR], lab.BINS)].argmax(axis=1)].mean())
fixed_best_idx = int(tab_rewards.mean(axis=0).argmax())
fixed_acc = float(test_scores[:, fixed_best_idx].mean())
random_acc = float(test_scores.mean())
print(f"1-Step Benchmark -> Tabular Q: {tab_acc:.4f} | Best Fixed Dose ({lab.DOSES[lab.ACTIONS[fixed_best_idx]]} ppm): {fixed_acc:.4f} | Uniform Random: {random_acc:.4f}")


class SequentialAssayMDP:
    """Multi-step enzyme assay MDP with Bayesian log-posterior state & Fisher D-opt shaping."""
    def __init__(self, ref_rates: np.ndarray, ref_fam: np.ndarray, budget: int = 2, seed: int = 123):
        self.ref_rates = ref_rates
        self.ref_fam = ref_fam
        self.budget = budget
        self.rng = np.random.default_rng(seed)
        self.centers = lab.CENTERS  # (4, 2): [Vmax, Km]

    def _obs_state(self) -> np.ndarray:
        # 12-d continuous state: [anchor_norm, mask_5, last_rates_norm_5, step_frac]
        s = np.zeros(12, dtype=np.float32)
        s[0] = (self.rates[lab.ANCHOR] - 100.0) / 60.0
        s[1:6] = self.visited.astype(np.float32)
        s[6:11] = ((self.observed_extra - 100.0) / 60.0) * self.visited
        s[11] = self.step_idx / max(1, self.budget)
        return s

    def reset(self) -> np.ndarray:
        self.fam = int(self.rng.integers(len(self.centers)))
        self.params = self.centers[self.fam] * np.exp(self.rng.normal(0.0, [0.13, 0.23]))
        vmax, km = self.params
        self.rates = vmax * lab.DOSES / (km + lab.DOSES) + self.rng.normal(0.0, 11.0, size=len(lab.DOSES))
        self.visited = np.zeros(len(lab.ACTIONS), dtype=bool)
        self.observed_extra = np.zeros(len(lab.ACTIONS), dtype=np.float32)
        self.step_idx = 0
        self.fisher = np.zeros((2, 2), dtype=np.float64)
        self._accum_fisher(lab.DOSES[lab.ANCHOR])
        return self._obs_state()

    def _accum_fisher(self, dose: float) -> float:
        # Local Fisher information matrix evaluated around geometric center prior
        v0, k0 = 175.0, 0.11
        dv = dose / (k0 + dose)
        dk = -v0 * dose / ((k0 + dose) ** 2)
        j = np.array([dv, dk]) / 11.0
        prev_logdet = math.log(max(1e-8, np.linalg.det(self.fisher + 1e-3 * np.eye(2))))
        self.fisher += np.outer(j, j)
        new_logdet = math.log(max(1e-8, np.linalg.det(self.fisher + 1e-3 * np.eye(2))))
        return max(0.0, new_logdet - prev_logdet)

    def decode_family(self) -> int:
        cols = [lab.ANCHOR] + [int(lab.ACTIONS[i]) for i in np.flatnonzero(self.visited)]
        centers = np.stack([self.ref_rates[self.ref_fam == f][:, cols].mean(axis=0) for f in range(4)])
        vars_ = np.stack([self.ref_rates[self.ref_fam == f][:, cols].var(axis=0) for f in range(4)])
        obs = np.array([self.rates[c] for c in cols])[None, :]
        loss = (((obs - centers) ** 2) / vars_ + np.log(vars_)).sum(axis=1)
        return int(np.argmin(loss))

    def step(self, action_idx: int) -> tuple[np.ndarray, float, bool]:
        repeat_penalty = -0.15 if self.visited[action_idx] else 0.0
        self.visited[action_idx] = True
        dose = float(lab.DOSES[lab.ACTIONS[action_idx]])
        self.observed_extra[action_idx] = float(self.rates[lab.ACTIONS[action_idx]])
        info_gain = self._accum_fisher(dose)
        self.step_idx += 1
        done = self.step_idx >= self.budget
        terminal_acc = float(self.decode_family() == self.fam) if done else 0.0
        reward = terminal_acc + 0.08 * info_gain + repeat_penalty
        return self._obs_state(), reward, done

In [ ]:
# Cell 7 · PyTorch Double DQN with Experience Replay Buffer & Polyak Target Synchronization
class QNetwork(nn.Module):
    def __init__(self, state_dim: int = 12, n_actions: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 96), nn.LayerNorm(96), nn.SiLU(),
            nn.Linear(96, 96), nn.LayerNorm(96), nn.SiLU(),
            nn.Linear(96, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity: int = 15000, state_dim: int = 12):
        self.cap = capacity
        self.ptr = 0
        self.size = 0
        self.s = np.zeros((capacity, state_dim), dtype=np.float32)
        self.a = np.zeros(capacity, dtype=np.int64)
        self.r = np.zeros(capacity, dtype=np.float32)
        self.ns = np.zeros((capacity, state_dim), dtype=np.float32)
        self.d = np.zeros(capacity, dtype=np.float32)

    def push(self, s, a, r, ns, d):
        self.s[self.ptr], self.a[self.ptr], self.r[self.ptr], self.ns[self.ptr], self.d[self.ptr] = s, a, r, ns, float(d)
        self.ptr = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, bs: int):
        idx = np.random.randint(0, self.size, size=bs)
        return (
            torch.tensor(self.s[idx], device=DEVICE),
            torch.tensor(self.a[idx], device=DEVICE),
            torch.tensor(self.r[idx], device=DEVICE),
            torch.tensor(self.ns[idx], device=DEVICE),
            torch.tensor(self.d[idx], device=DEVICE),
        )


def train_double_dqn(episodes: int = 2400, budget: int = 2):
    env = SequentialAssayMDP(rates_tr, fam_tr, budget=budget, seed=99)
    q_online = QNetwork().to(DEVICE)
    q_target = QNetwork().to(DEVICE)
    q_target.load_state_dict(q_online.state_dict())
    opt = torch.optim.Adam(q_online.parameters(), lr=1.5e-3)
    buf = ReplayBuffer()
    gamma, tau_polyak = 0.96, 0.02
    dqn_history = []

    for ep in range(episodes):
        eps = max(0.05, 0.65 - 0.60 * (ep / episodes))
        s = env.reset()
        ep_ret = 0.0
        done = False
        while not done:
            if np.random.random() < eps:
                a = int(np.random.randint(0, len(lab.ACTIONS)))
            else:
                with torch.no_grad():
                    a = int(q_online(torch.tensor(s, device=DEVICE).unsqueeze(0)).argmax(dim=1).item())
            ns, r, done = env.step(a)
            buf.push(s, a, r, ns, done)
            s = ns
            ep_ret += r
            if buf.size >= 256:
                bs_s, bs_a, bs_r, bs_ns, bs_d = buf.sample(128)
                with torch.no_grad():
                    best_next = q_online(bs_ns).argmax(dim=1, keepdim=True)
                    next_q = q_target(bs_ns).gather(1, best_next).squeeze(1)
                    target_y = bs_r + gamma * (1.0 - bs_d) * next_q
                pred_q = q_online(bs_s).gather(1, bs_a.unsqueeze(1)).squeeze(1)
                loss = F.smooth_l1_loss(pred_q, target_y)
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(q_online.parameters(), 2.0)
                opt.step()
                for p_t, p_o in zip(q_target.parameters(), q_online.parameters()):
                    p_t.data.mul_(1.0 - tau_polyak).add_(tau_polyak * p_o.data)
        dqn_history.append(float(env.decode_family() == env.fam))
    return q_online, dqn_history


q_dqn, dqn_hist = train_double_dqn(episodes=2400, budget=2)
eval_env = SequentialAssayMDP(rates_tr, fam_tr, budget=2, seed=777)
dqn_accs, rand_accs_2step = [], []
for _ in range(800):
    s = eval_env.reset()
    done = False
    while not done:
        with torch.no_grad():
            a = int(q_dqn(torch.tensor(s, device=DEVICE).unsqueeze(0)).argmax(dim=1).item())
        s, _, done = eval_env.step(a)
    dqn_accs.append(float(eval_env.decode_family() == eval_env.fam))

    eval_env.reset()
    for a_rand in np.random.choice(len(lab.ACTIONS), size=2, replace=False):
        eval_env.step(int(a_rand))
    rand_accs_2step.append(float(eval_env.decode_family() == eval_env.fam))

print(f"2-Step Sequential Evaluation (n=800): Double DQN = {np.mean(dqn_accs):.4f} vs Random Distinct 2-Dose = {np.mean(rand_accs_2step):.4f}")

In [ ]:
# Cell 8 · Continuous-Dose PPO-Clip (Beta Policy Head + GAE-lambda + Fisher D-Optimality)
class ContinuousAssayEnv:
    """Continuous substrate concentration design a in [0.02, 1.20] ppm."""
    def __init__(self, budget: int = 2, seed: int = 314):
        self.budget = budget
        self.rng = np.random.default_rng(seed)

    def reset(self) -> np.ndarray:
        self.fam = int(self.rng.integers(4))
        self.params = lab.CENTERS[self.fam] * np.exp(self.rng.normal(0.0, [0.13, 0.23]))
        self.history = [0.11, float(self._measure(0.11))]
        self.doses_picked = [0.11]
        self.fisher = np.zeros((2, 2))
        self._info_step(0.11)
        self.step_idx = 0
        return self._state()

    def _measure(self, conc: float) -> float:
        v, k = self.params
        return float(v * conc / (k + conc) + self.rng.normal(0.0, 11.0))

    def _info_step(self, conc: float) -> float:
        v0, k0 = 175.0, 0.11
        j = np.array([conc / (k0 + conc), -v0 * conc / ((k0 + conc) ** 2)]) / 11.0
        before = math.log(max(1e-8, np.linalg.det(self.fisher + 1e-3 * np.eye(2))))
        self.fisher += np.outer(j, j)
        return max(0.0, math.log(max(1e-8, np.linalg.det(self.fisher + 1e-3 * np.eye(2)))) - before)

    def _state(self) -> np.ndarray:
        arr = np.zeros(8, dtype=np.float32)
        for idx, val in enumerate(self.history[:6]):
            arr[idx] = val / 150.0 if idx % 2 else val
        arr[6] = self.step_idx / self.budget
        arr[7] = math.log(max(1e-6, np.linalg.det(self.fisher + 1e-3 * np.eye(2)))) / 10.0
        return arr

    def posterior_accuracy(self) -> float:
        # Bayesian MAP over 4 family Gaussians given all continuous doses observed
        logp = np.zeros(4)
        xs = np.array(self.history[0::2])
        ys = np.array(self.history[1::2])
        for f, (vc, kc) in enumerate(lab.CENTERS):
            pred = vc * xs / (kc + xs)
            logp[f] = -0.5 * np.sum(((ys - pred) / 22.0) ** 2)
        return float(int(np.argmax(logp)) == self.fam)

    def step(self, a_unit: float) -> tuple[np.ndarray, float, bool]:
        conc = 0.02 + float(np.clip(a_unit, 0.0, 1.0)) * 1.18
        rate = self._measure(conc)
        self.history.extend([conc, rate])
        self.doses_picked.append(conc)
        gain = self._info_step(conc)
        self.step_idx += 1
        done = self.step_idx >= self.budget
        acc = self.posterior_accuracy() if done else 0.0
        return self._state(), acc + 0.06 * gain, done


class PPOActorCritic(nn.Module):
    def __init__(self, state_dim: int = 8):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(state_dim, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.alpha_head = nn.Linear(64, 1)
        self.beta_head = nn.Linear(64, 1)
        self.val_head = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor):
        h = self.shared(x)
        alpha = F.softplus(self.alpha_head(h)).squeeze(-1) + 1.05
        beta = F.softplus(self.beta_head(h)).squeeze(-1) + 1.05
        val = self.val_head(h).squeeze(-1)
        return alpha, beta, val


def train_ppo(iterations: int = 45, batch_eps: int = 128):
    env = ContinuousAssayEnv(budget=2, seed=2026)
    ac = PPOActorCritic().to(DEVICE)
    opt = torch.optim.Adam(ac.parameters(), lr=2.5e-3)
    gamma, gae_lam, clip_eps = 0.98, 0.95, 0.2
    ppo_curve = []

    for _ in range(iterations):
        states, acts, logps, rets, advs, ep_accs = [], [], [], [], [], []
        for _ in range(batch_eps):
            ep_s, ep_a, ep_lp, ep_r, ep_v = [], [], [], [], []
            s = env.reset()
            done = False
            while not done:
                st = torch.tensor(s, device=DEVICE).unsqueeze(0)
                with torch.no_grad():
                    al, be, v = ac(st)
                    dist = Beta(al, be)
                    a = dist.sample().clamp(1e-4, 1 - 1e-4)
                    lp = dist.log_prob(a)
                ns, r, done = env.step(float(a.item()))
                ep_s.append(s); ep_a.append(float(a.item())); ep_lp.append(float(lp.item())); ep_r.append(r); ep_v.append(float(v.item()))
                s = ns
            ep_accs.append(env.posterior_accuracy())
            # GAE-lambda returns & advantages
            gae, next_v = 0.0, 0.0
            for t in reversed(range(len(ep_r))):
                delta = ep_r[t] + gamma * next_v - ep_v[t]
                gae = delta + gamma * gae_lam * gae
                next_v = ep_v[t]
                states.append(ep_s[t]); acts.append(ep_a[t]); logps.append(ep_lp[t])
                advs.append(gae); rets.append(gae + ep_v[t])

        st_t = torch.tensor(np.array(states), device=DEVICE)
        act_t = torch.tensor(acts, dtype=torch.float32, device=DEVICE)
        old_lp_t = torch.tensor(logps, dtype=torch.float32, device=DEVICE)
        ret_t = torch.tensor(rets, dtype=torch.float32, device=DEVICE)
        adv_t = torch.tensor(advs, dtype=torch.float32, device=DEVICE)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-6)

        for _ in range(4):
            al, be, v = ac(st_t)
            dist = Beta(al, be)
            new_lp = dist.log_prob(act_t)
            ratio = torch.exp(new_lp - old_lp_t)
            surr1 = ratio * adv_t
            surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv_t
            loss = -torch.min(surr1, surr2).mean() + 0.5 * F.mse_loss(v, ret_t) - 0.01 * dist.entropy().mean()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ac.parameters(), 1.0)
            opt.step()
        ppo_curve.append(float(np.mean(ep_accs)))
    return ac, ppo_curve


ppo_model, ppo_hist = train_ppo(iterations=45, batch_eps=128)
print(f"PPO Continuous-Dose Active Design Final Acc: {ppo_hist[-1]:.4f} (start={ppo_hist[0]:.4f})")

## Chapter 4 · Unified Multi-Panel Figure Suite & Direct Export to `studio/workspace.html`

Below we render a 4-panel comparative visual diagnostic chart summarizing **Kinetics Accuracy + OOD RMSE**, **Biomarker Attention Saliency**, **AlphaFold pLDDT Calibration**, and **Tabular Q vs Double DQN vs Continuous PPO RL Learning Curves**, and export a live **`assay-evidence/1`** JSON packet that can be imported directly into `Evidence Workspace`.

In [ ]:
# Cell 9 · Publication Multi-Panel Visual Diagnostics + Workspace JSON Packet Export
fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=130)

# Panel A: Classical ML vs PyTorch KinResNet (Vmax MAE & OOD 2.5 ppm RMSE)
ax = axes[0, 0]
all_kin = pd.DataFrame(kin_results + [kinresnet_row])
x_pos = np.arange(len(all_kin))
ax.bar(x_pos - 0.18, all_kin["Vmax MAE"], width=0.35, label="Vmax MAE (in-domain)", color="#175c55")
ax.bar(x_pos + 0.18, all_kin["OOD [2.5 ppm] RMSE"], width=0.35, label="OOD 2.5 ppm RMSE", color="#d96b27")
ax.set_xticks(x_pos)
ax.set_xticklabels([m.split("(")[0].strip() for m in all_kin["Model"]], rotation=14, ha="right", fontsize=9)
ax.set_title("A · Enzyme Parameter Inference & OOD Generalization")
ax.set_ylabel("Error (counts/min/min)")
ax.legend(frameon=False, fontsize=8)

# Panel B: Golub Attention Biomarker Attribution Top 8
ax = axes[0, 1]
top_names = [panel_rows[k][0] for k in top_saliency][::-1]
top_vals = saliency[top_saliency][::-1]
ax.barh(top_names, top_vals, color="#0053d6")
ax.set_title("B · PyTorch Self-Attention Golub Gene Saliency")
ax.set_xlabel("Mean |Input x Gradient| Attribution")

# Panel C: Conservative Neural Force Field Verlet Hamiltonian Drift
ax = axes[1, 0]
ax.plot(np.arange(len(energies)) * dt, energies, color="#10946d", lw=2)
ax.set_title(f"C · SmoothNet Conservative Verlet Energy (|ΔH|max={e_drift:.4f})")
ax.set_xlabel("Simulation Time (a.u.)")
ax.set_ylabel("Hamiltonian H(x, v)")

# Panel D: Sequential RL Learning Curves (PPO Continuous vs Double DQN)
ax = axes[1, 1]
dqn_smooth = pd.Series(dqn_hist).rolling(120, min_periods=20).mean()
ax.plot(np.linspace(0, 100, len(dqn_smooth)), dqn_smooth, label="PyTorch Double DQN (2-step discrete)", color="#8b5cf6")
ax.plot(np.linspace(0, 100, len(ppo_hist)), ppo_hist, "o-", ms=3, label="PyTorch PPO-Clip (continuous dose)", color="#e11d48")
ax.axhline(tab_acc, color="#0ea5e9", ls="--", label=f"1-Step Tabular Q ({tab_acc:.3f})")
ax.axhline(random_acc, color="#64748b", ls=":", label=f"1-Step Random ({random_acc:.3f})")
ax.set_title("D · Reinforcement Learning Sequential Assay Discovery")
ax.set_xlabel("Training Progress (%)")
ax.set_ylabel("Held-Out Family Accuracy")
ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
fig.savefig(ROOT / "ml" / "bioai_suite_summary.png", dpi=150)
plt.show()

# Export schema-valid assay-evidence/1 JSON packet compatible with studio/workspace.html
packet = {
    "schema": "assay-evidence/1",
    "note": "Benchmarks generated by ml/BioAI_PyTorch_ML_RL_Suite.ipynb across PyTorch KinResNet, GolubAttentionNet, and Sequential DQN/PPO RL.",
    "runs": [
        {
            "id": "nb-kinresnet-ood-25",
            "savedAt": "2026-11-29T00:00:00.000Z",
            "kind": "simulated",
            "title": "PyTorch KinResNet (OOD 2.5 ppm RMSE)",
            "dataset": "Michaelis-Menten 6-dose synthetic benchmark (n=800)",
            "method": "Heteroscedastic Residual MLP + GaussianNLL + MC Dropout",
            "metric": "OOD endpoint RMSE",
            "value": round(float(kinresnet_row["OOD [2.5 ppm] RMSE"]), 4),
            "unit": "counts/min/min",
            "source": "https://github.com/tun539/assay-evidence-lab/blob/main/ml/BioAI_PyTorch_ML_RL_Suite.ipynb",
            "details": {"comparisonKey": "bioai-kinetics-ood", "vmaxMae": round(float(kinresnet_row["Vmax MAE"]), 4), "kmMae": round(float(kinresnet_row["Km MAE"]), 4)},
        },
        {
            "id": "nb-rl-ppo-seq",
            "savedAt": "2026-11-29T00:01:00.000Z",
            "kind": "simulated",
            "title": "Continuous PPO-Clip Active Assay Design",
            "dataset": "Sequential Assay Design MDP (budget=2 doses)",
            "method": "PPO-Clip Beta Policy + GAE-lambda + Fisher D-Opt Shaping",
            "metric": "Family identification accuracy",
            "value": round(float(ppo_hist[-1]), 4),
            "unit": "proportion",
            "source": "https://github.com/tun539/assay-evidence-lab/blob/main/ml/BioAI_PyTorch_ML_RL_Suite.ipynb",
            "details": {"comparisonKey": "bioai-rl-assay", "dqn2Step": round(float(np.mean(dqn_accs)), 4), "tabular1Step": round(float(tab_acc), 4)},
        },
    ],
}
out_pkt = ROOT / "ml" / "workspace_ai_benchmark_packet.json"
out_pkt.write_text(json.dumps(packet, indent=2) + "\n", encoding="utf-8")
print("Exported schema-valid workspace packet ->", out_pkt)